# 00 Build Flat Processed OOC Dataset

This notebook rebuilds a **flat, self-contained processed dataset** with:

- **all local images in one single folder**: `data/processed/ooc_pairs/images/`
- normalized CSV manifests in: `data/processed/ooc_pairs/manifests/`
- a validation report in: `data/processed/ooc_pairs/reports/`

## Design choice

This notebook intentionally keeps the processed image layer **flat**.

That means files will look like:

- `data/processed/ooc_pairs/images/verite_true_0.jpg`
- `data/processed/ooc_pairs/images/mmfakebench_test_real_bbc_test_500_bbc_test_0.png`

instead of nested folders.

This works **as long as image filenames are made unique**. So the notebook creates a unique `image_id` and a unique processed filename for each copied image.

## Important

- **NewsCLIPpings** remains embedding-only, because local VisualNews images are not available.
- **VERITE** and **MMFakeBench** local images are copied into the flat processed folder.
- Contexts/labels are best kept in **manifest CSVs**, not as separate text files.


In [1]:
from pathlib import Path
import json
import shutil
import re
import pandas as pd

cwd = Path.cwd()
if (cwd / 'data').exists() and (cwd / 'notebooks').exists():
    ROOT = cwd
else:
    ROOT = cwd.parent

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed' / 'ooc_pairs'
IMAGES = PROCESSED / 'images'
MANIFESTS = PROCESSED / 'manifests'
REPORTS = PROCESSED / 'reports'
SPLITS_DIR = ROOT / 'data' / 'splits'

NEWS_ROOT = RAW / 'newsclippings'
VERITE_ROOT = RAW / 'verite' / 'image-text-verification' / 'VERITE'
VERITE_CSV = VERITE_ROOT / 'VERITE.csv'
VERITE_IMG_RAW = VERITE_ROOT / 'images'
MMFAKE_ROOT = RAW / 'mmfakebench'

for p in [IMAGES, MANIFESTS, REPORTS, SPLITS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.gif']

REBUILD_FROM_SCRATCH = True

print('ROOT:', ROOT)
print('Processed images folder:', IMAGES)
print('Manifest folder:', MANIFESTS)
print('Future splits folder:', SPLITS_DIR)


ROOT: e:\eve-rag
Processed images folder: e:\eve-rag\data\processed\ooc_pairs\images
Manifest folder: e:\eve-rag\data\processed\ooc_pairs\manifests
Future splits folder: e:\eve-rag\data\splits


## Optional cleanup

This cell removes old processed images/manifests/reports and rebuilds them from scratch.

Set `REBUILD_FROM_SCRATCH = False` in the setup cell if you do not want deletion.


In [2]:
if REBUILD_FROM_SCRATCH:
    for folder in [IMAGES, MANIFESTS, REPORTS]:
        if folder.exists():
            for item in folder.iterdir():
                if item.is_dir():
                    shutil.rmtree(item)
                else:
                    item.unlink()
        folder.mkdir(parents=True, exist_ok=True)
    print('Cleared processed images, manifests, and reports.')
else:
    print('Skipping cleanup.')


Cleared processed images, manifests, and reports.


In [3]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_records(path: Path):
    obj = load_json(path)

    if isinstance(obj, list):
        return obj

    if isinstance(obj, dict):
        if "annotations" in obj:
            ann = obj["annotations"]
            if isinstance(ann, list):
                return ann
            if isinstance(ann, dict):
                return list(ann.values())

        for key in ["data", "items", "samples", "examples", "pairs"]:
            if key in obj:
                val = obj[key]
                if isinstance(val, list):
                    return val
                if isinstance(val, dict):
                    return list(val.values())

        if all(isinstance(v, dict) for v in obj.values()):
            rows = []
            for k, v in obj.items():
                row = dict(v)
                row.setdefault("_key", k)
                rows.append(row)
            return rows

        if all(isinstance(v, list) for v in obj.values()):
            return pd.DataFrame(obj).to_dict("records")

    raise ValueError(f"Unsupported JSON structure: {path}")


def safe_str(x):
    if x is None:
        return ""
    if not isinstance(x, (list, dict)):
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass
    return str(x)


def norm_label(value):
    s = safe_str(value).strip().lower()

    if s in {"0", "true", "real", "matched", "match", "pristine", "truthful", "yes", "original"}:
        return 0

    if s in {"1", "false", "fake", "falsified", "ooc", "out-of-context", "mismatch", "miscaptioned", "no"}:
        return 1

    return None


def label_name(label):
    if label == 0:
        return "match"
    if label == 1:
        return "mismatch"
    return "unknown"


def relpath(p: Path):
    if p is None:
        return ""
    return str(p.relative_to(ROOT)).replace("\\", "/")


def sanitize_token(s: str):
    s = safe_str(s).strip().replace("\\", "/").replace(" ", "_")
    s = Path(s).name if "/" in s else s
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "item"


def find_by_name_or_stem(folder: Path, value):
    s = safe_str(value).strip()

    if not s or s.lower() == "nan":
        return None

    name = Path(s).name
    stem = Path(name).stem

    candidates = [folder / name]

    for ext in IMAGE_EXTS:
        candidates.append(folder / f"{stem}{ext}")

    try:
        n = int(float(stem))
        for width in [1, 4, 5, 6]:
            for ext in IMAGE_EXTS:
                candidates.append(folder / f"{n:0{width}d}{ext}")
    except Exception:
        pass

    for p in candidates:
        if p.exists():
            return p

    return None


def copy_image_flat(src: Path, unique_stem: str):
    if src is None or not src.exists():
        return None

    ext = src.suffix.lower()

    if ext not in IMAGE_EXTS:
        return None

    dst = IMAGES / f"{unique_stem}{ext}"

    if not dst.exists():
        shutil.copy2(src, dst)

    return dst


BASE_COLUMNS = [
    "pair_id", "dataset", "split",
    "image_id", "image_path", "raw_image_path", "has_image",
    "caption", "label", "label_name",
    "source_image_id", "source_text_id", "original_id",
    "clip_image_key", "clip_text_key",
    "text_source", "image_source", "fake_cls",
    "topic", "source", "timestamp", "title",
    "raw_label", "notes",
]


def finalize(df: pd.DataFrame):
    for col in BASE_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    df = df[BASE_COLUMNS].copy()

    for col in df.columns:
        if col not in {"label", "has_image"}:
            df[col] = df[col].fillna("").astype(str)

    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
    df["has_image"] = pd.to_numeric(df["has_image"], errors="coerce").fillna(0).astype(int)

    return df

## Build NewsCLIPpings manifests

These rows remain embedding-only. We keep:

- `raw_image_path`
- `clip_image_key`
- `clip_text_key`

but `image_path` stays blank and `has_image = 0`.


In [4]:
def build_news_metadata(split_name):
    meta_path = NEWS_ROOT / 'metadata' / f'{split_name}.json'
    if not meta_path.exists():
        return {}
    raw = load_json(meta_path)
    index = {}
    if isinstance(raw, dict):
        iterator = raw.items()
    else:
        iterator = [(None, row) for row in load_records(meta_path)]
    for key, row in iterator:
        if not isinstance(row, dict):
            continue
        keys = {
            safe_str(key),
            safe_str(row.get('id', '')),
            safe_str(row.get('image_id', '')),
            Path(safe_str(row.get('image_path', ''))).name,
            Path(safe_str(row.get('image_path', ''))).stem,
        }
        for k in keys:
            if k and k != 'nan':
                index[k] = row
                try:
                    index[str(int(float(k)))] = row
                except Exception:
                    pass
    return index


def build_news(split_name):
    pairs_path = NEWS_ROOT / f'{split_name}.json'
    if not pairs_path.exists():
        print(f'[WARN] Missing NewsCLIPpings pairs: {pairs_path}')
        return pd.DataFrame(columns=BASE_COLUMNS)

    pairs = load_records(pairs_path)
    meta = build_news_metadata(split_name)
    rows = []

    for idx, r in enumerate(pairs):
        pair_id_raw = r.get('id', idx)
        image_id_raw = r.get('image_id', r.get('image', r.get('image_path', '')))
        y = int(r.get('falsified', r.get('label', 0)))

        m = meta.get(safe_str(image_id_raw)) or meta.get(safe_str(pair_id_raw))
        caption = raw_image_path = topic = source = timestamp = title = ''
        if m:
            caption = safe_str(m.get('caption', ''))
            raw_image_path = safe_str(m.get('image_path', ''))
            topic = safe_str(m.get('topic', ''))
            source = safe_str(m.get('source', ''))
            timestamp = safe_str(m.get('timestamp', ''))
            title = safe_str(m.get('title', ''))

        rows.append({
            'pair_id': f'nc_{pair_id_raw}_{y}',
            'dataset': 'newsclippings',
            'split': split_name,
            'image_id': f'ncimg_{sanitize_token(image_id_raw)}',
            'image_path': '',
            'raw_image_path': raw_image_path,
            'has_image': 0,
            'caption': caption,
            'label': y,
            'label_name': label_name(y),
            'source_image_id': image_id_raw,
            'source_text_id': pair_id_raw,
            'original_id': pair_id_raw,
            'clip_image_key': image_id_raw,
            'clip_text_key': pair_id_raw,
            'topic': topic,
            'source': source,
            'timestamp': timestamp,
            'title': title,
            'raw_label': y,
            'notes': 'embedding-only row; local VisualNews image unavailable'
        })

    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / {'train':'train.csv','val':'val.csv','test':'test.csv'}[split_name], index=False, encoding='utf-8')
    return out


## Build VERITE manifest and copy images into the single flat folder

All copied VERITE images are placed directly under `data/processed/ooc_pairs/images/`.

To avoid collisions, the filename is prefixed, for example:

- `verite_true_0.jpg`
- `verite_false_0.jpg`
- `verite_0007.webp`


In [5]:
def build_verite():
    if not VERITE_CSV.exists():
        print(f'[WARN] Missing VERITE CSV: {VERITE_CSV}')
        return pd.DataFrame(columns=BASE_COLUMNS)

    df = pd.read_csv(VERITE_CSV)
    rows = []
    copied = 0

    for i, r in df.iterrows():
        raw_image_path = safe_str(r.get('image_path', '')).strip()
        caption = safe_str(r.get('caption', r.get('text', ''))).strip()
        raw_label = r.get('label', '')
        y = norm_label(raw_label)

        candidates = []
        p = find_by_name_or_stem(VERITE_IMG_RAW, raw_image_path)
        if p:
            candidates.append(p)
        for stem in [str(i), f'{i:04d}', f'{i+1:04d}', f'{i:05d}', f'{i+1:05d}']:
            p = find_by_name_or_stem(VERITE_IMG_RAW, stem)
            if p:
                candidates.append(p)

        found = candidates[0] if candidates else None
        processed_img = None
        image_id = ''
        if found:
            unique_stem = f'verite_{sanitize_token(found.stem)}'
            processed_img = copy_image_flat(found, unique_stem)
            image_id = unique_stem
            if processed_img is not None:
                copied += 1

        rows.append({
            'pair_id': f'verite_{i:04d}',
            'dataset': 'verite',
            'split': 'test',
            'image_id': image_id,
            'image_path': relpath(processed_img) if processed_img else '',
            'raw_image_path': raw_image_path,
            'has_image': int(processed_img is not None),
            'caption': caption,
            'label': y,
            'label_name': label_name(y),
            'source_image_id': Path(raw_image_path).stem,
            'source_text_id': safe_str(r.get('Unnamed: 0', i)),
            'original_id': safe_str(r.get('Unnamed: 0', i)),
            'clip_image_key': safe_str(r.get('Unnamed: 0', i)),
            'clip_text_key': safe_str(r.get('Unnamed: 0', i)),
            'raw_label': raw_label,
            'notes': 'copied to flat processed image folder' if processed_img else 'missing local image; row kept for accounting'
        })

    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / 'verite_test.csv', index=False, encoding='utf-8')
    print('VERITE copied/resolved image rows:', copied)
    return out


## Build MMFakeBench manifest and copy images into the single flat folder

MMFakeBench also goes into the same single image folder, but filenames are prefixed to stay unique, for example:

- `mmfakebench_test_real_bbc_test_500_bbc_test_0.png`
- `mmfakebench_val_fake_some_folder_xxx.jpg`


In [6]:
def resolve_mmfake_image(img_rel, split_name):
    s = safe_str(img_rel).strip()

    if not s or s.lower() == "nan":
        return None

    s_clean = s.replace("\\", "/").lstrip("/")
    name = Path(s_clean).name

    candidates = [
        MMFAKE_ROOT / s_clean,
        MMFAKE_ROOT / f"MMFakeBench_{split_name}" / s_clean,
        MMFAKE_ROOT / f"MMFakeBench_{split_name}" / name,
    ]

    for p in candidates:
        if p.exists():
            return p

    split_dir = MMFAKE_ROOT / f"MMFakeBench_{split_name}"

    if split_dir.exists() and name:
        matches = list(split_dir.rglob(name))
        if matches:
            return matches[0]

    return None


def build_mmfake(split_name):
    json_path = MMFAKE_ROOT / f"MMFakeBench_{split_name}.json"

    if not json_path.exists():
        print(f"[WARN] Missing MMFakeBench JSON: {json_path}")
        return pd.DataFrame(columns=BASE_COLUMNS)

    records = load_records(json_path)
    rows = []
    copied = 0

    for i, r in enumerate(records):
        caption = safe_str(r.get("text", r.get("caption", r.get("claim", "")))).strip()
        raw_image_path = safe_str(r.get("image_path", r.get("image", r.get("img_path", "")))).strip()
        raw_label = r.get("gt_answers", r.get("label", ""))

        y = norm_label(raw_label)
        if y is None:
            y = 1 if "/fake/" in raw_image_path.replace("\\", "/").lower() else 0

        raw_img = resolve_mmfake_image(raw_image_path, split_name)

        processed_img = None
        image_id = ""

        if raw_img:
            s_clean = raw_image_path.replace("\\", "/").lstrip("/")
            rel_token = sanitize_token(s_clean.replace("/", "_"))
            unique_stem = f"mmfakebench_{split_name}_{Path(rel_token).stem}"
            processed_img = copy_image_flat(raw_img, unique_stem)
            image_id = unique_stem

            if processed_img is not None:
                copied += 1

        rows.append({
            "pair_id": f"mmfakebench_{split_name}_{i:05d}",
            "dataset": "mmfakebench",
            "split": split_name,
            "image_id": image_id,
            "image_path": relpath(processed_img) if processed_img else "",
            "raw_image_path": raw_image_path,
            "has_image": int(processed_img is not None),
            "caption": caption,
            "label": y,
            "label_name": label_name(y),
            "source_image_id": Path(raw_image_path).stem,
            "source_text_id": i,
            "original_id": i,
            "text_source": safe_str(r.get("text_source", "")),
            "image_source": safe_str(r.get("image_source", "")),
            "fake_cls": safe_str(r.get("fake_cls", "")),
            "raw_label": raw_label,
            "notes": "copied to flat processed image folder" if processed_img else "missing local image",
        })

    out = finalize(pd.DataFrame(rows))
    out.to_csv(MANIFESTS / f"mmfakebench_{split_name}.csv", index=False, encoding="utf-8")

    print(f"MMFakeBench {split_name} copied/resolved image rows:", copied)

    return out

## Build everything


In [7]:
outputs = []

print('Building NewsCLIPpings...')
for split in ['train', 'val', 'test']:
    df = build_news(split)
    outputs.append(df)
    print(f"  {split}: rows={len(df)}, captions={(df['caption'].str.len() > 0).sum()}, has_image={df['has_image'].sum()}")

print('Building VERITE...')
verite_df = build_verite()
outputs.append(verite_df)
print(f"  verite_test: rows={len(verite_df)}, has_image={verite_df['has_image'].sum()}")

print('Building MMFakeBench...')
for split in ['val', 'test']:
    df = build_mmfake(split)
    outputs.append(df)
    print(f"  mmfakebench_{split}: rows={len(df)}, has_image={df['has_image'].sum()}")

all_df = finalize(pd.concat(outputs, ignore_index=True))
all_df.to_csv(MANIFESTS / 'all.csv', index=False, encoding='utf-8')

print('Saved all.csv:', len(all_df))
print('Wrote manifests to:', MANIFESTS)
print('Wrote flat processed images to:', IMAGES)


Building NewsCLIPpings...
  train: rows=71072, captions=71072, has_image=0
  val: rows=7024, captions=7024, has_image=0
  test: rows=7264, captions=7264, has_image=0
Building VERITE...
VERITE copied/resolved image rows: 621
  verite_test: rows=1001, has_image=621
Building MMFakeBench...
MMFakeBench val copied/resolved image rows: 1000
  mmfakebench_val: rows=1000, has_image=1000
MMFakeBench test copied/resolved image rows: 10000
  mmfakebench_test: rows=10000, has_image=10000
Saved all.csv: 97361
Wrote manifests to: e:\eve-rag\data\processed\ooc_pairs\manifests
Wrote flat processed images to: e:\eve-rag\data\processed\ooc_pairs\images


## Validate


In [8]:
summary_rows = []
for csv_name in ['train.csv', 'val.csv', 'test.csv', 'verite_test.csv', 'mmfakebench_val.csv', 'mmfakebench_test.csv', 'all.csv']:
    path = MANIFESTS / csv_name
    df = pd.read_csv(path, dtype={'image_path':'string', 'caption':'string', 'pair_id':'string', 'image_id':'string'}, low_memory=False)
    broken = df[(df['has_image'] == 1) & (~df['image_path'].fillna('').map(lambda p: (ROOT / p).exists()))]
    summary_rows.append({
        'file': csv_name,
        'rows': len(df),
        'label_0': int((df['label'] == 0).sum()),
        'label_1': int((df['label'] == 1).sum()),
        'has_image': int(df['has_image'].sum()),
        'unique_image_ids': int(df['image_id'].replace('', pd.NA).dropna().nunique()),
        'empty_captions': int((df['caption'].fillna('').astype(str).str.len() == 0).sum()),
        'broken_image_paths': len(broken),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS / 'manifest_summary.csv', index=False)
summary_df


,file,rows,label_0,label_1,has_image,unique_image_ids,empty_captions,broken_image_paths
0,train.csv,71072,35536,35536,0,59876,0,0
1,val.csv,7024,3512,3512,0,5943,0,0
2,test.csv,7264,3632,3632,0,6147,0,0
3,verite_test.csv,1001,338,663,621,495,0,0
4,mmfakebench_val.csv,1000,300,700,1000,1000,0,0
5,mmfakebench_test.csv,10000,3000,7000,10000,10000,0,0
6,all.csv,97361,46318,51043,11621,83461,0,0


In [9]:
flat_image_count = sum(1 for p in IMAGES.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print('Flat processed image count:', flat_image_count)
print('Sample processed filenames:')
for p in sorted([p for p in IMAGES.iterdir() if p.is_file()])[:20]:
    print(' ', p.name)


Flat processed image count: 11495
Sample processed filenames:
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_1.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_10.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_100.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_101.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_102.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_103.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_104.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_105.png
  mmfakebench_test_fake_antifact_image_generation_test_500_coco_antifact_image_edit_test_md_106.png
  mmfakebench_test_fake_antifact_image_ge

## Recommended project rule

Use this simple rule going forward:

- `data/raw/` = preserve original downloaded datasets
- `data/processed/ooc_pairs/images/` = one flat merged image folder
- `data/processed/ooc_pairs/manifests/` = all normalized metadata, labels, and text
- `data/splits/` = later experiment-specific split files if you want separate training recipes

This is a clean structure and is easy to work with.


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

MANIFESTS = ROOT / "data" / "processed" / "ooc_pairs" / "manifests"
IMAGES = ROOT / "data" / "processed" / "ooc_pairs" / "images"

for name in [
    "train.csv",
    "val.csv",
    "test.csv",
    "verite_test.csv",
    "mmfakebench_val.csv",
    "mmfakebench_test.csv",
    "all.csv",
]:
    df = pd.read_csv(MANIFESTS / name, dtype={"image_path": "string", "image_id": "string"}, low_memory=False)

    broken = df[
        (df["has_image"] == 1)
        & (~df["image_path"].fillna("").map(lambda p: (ROOT / p).exists()))
    ]

    print("\n", name)
    print("rows:", len(df))
    print("has_image:", int(df["has_image"].sum()))
    print("unique image_id:", df["image_id"].fillna("").replace("", pd.NA).dropna().nunique())
    print("broken image paths:", len(broken))

flat_count = sum(
    1 for p in IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".gif"]
)

print("\nFlat image files:", flat_count)


 train.csv
rows: 71072
has_image: 0
unique image_id: 59876
broken image paths: 0

 val.csv
rows: 7024
has_image: 0
unique image_id: 5943
broken image paths: 0

 test.csv
rows: 7264
has_image: 0
unique image_id: 6147
broken image paths: 0

 verite_test.csv
rows: 1001
has_image: 621
unique image_id: 495
broken image paths: 0

 mmfakebench_val.csv
rows: 1000
has_image: 1000
unique image_id: 1000
broken image paths: 0

 mmfakebench_test.csv
rows: 10000
has_image: 10000
unique image_id: 10000
broken image paths: 0

 all.csv
rows: 97361
has_image: 11621
unique image_id: 83461
broken image paths: 0

Flat image files: 11495
